In [ ]:
engine.fit(
    model=model,
    datamodule=datamodule,
)

In [ ]:
from anomalib.engine import Engine

engine = Engine()

print("Engine 创建成功！")
print(type(engine))

In [ ]:
engine.fit(
    model=model,
    datamodule=datamodule,
)


In [ ]:
import matplotlib.pyplot as plt

# 取出这一张图片的四种信息
img = target.image[target_index].detach().cpu()
gt_mask = target.gt_mask[target_index].detach().cpu()
anomaly_map = target.anomaly_map[target_index].detach().cpu()
pred_mask = target.pred_mask[target_index].detach().cpu()

# PyTorch 图片通常是 [C, H, W]
# matplotlib 需要 [H, W, C]
img = img.permute(1, 2, 0)

# 去掉 mask / anomaly map 多余的通道维度
gt_mask = gt_mask.squeeze()
anomaly_map = anomaly_map.squeeze()
pred_mask = pred_mask.squeeze()

# 画图
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img)
axes[0].set_title("Original Image")

axes[1].imshow(gt_mask, cmap="gray")
axes[1].set_title("Ground Truth")

heatmap = axes[2].imshow(anomaly_map, cmap="jet")
axes[2].set_title("PatchCore Anomaly Map")
plt.colorbar(heatmap, ax=axes[2], fraction=0.046)

axes[3].imshow(pred_mask, cmap="gray")
axes[3].set_title("Predicted Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
target.image[target_index]
target.gt_mask[target_index]
target.anomaly_map[target_index]
target.pred_mask[target_index]

In [ ]:
target = None
target_index = None

for batch in predictions:
    for i, path in enumerate(batch.image_path):
        if "broken_large" in str(path):
            target = batch
            target_index = i
            print("找到异常图片：", path)
            print("预测异常分数：", batch.pred_score[i].item())
            print("预测标签：", batch.pred_label[i].item())
            break

    if target is not None:
        break

In [ ]:
batch = predictions[0]
print(batch.keys())

In [ ]:
predictions = engine.predict(
    model=model,
    datamodule=datamodule,
)

print("预测完成！")
print("返回的 batch 数量：", len(predictions))
print("第一个 batch 类型：", type(predictions[0]))

In [ ]:
test_results = engine.test(
    model=model,
    datamodule=datamodule,
)

print("测试完成！")

In [ ]:
print("Memory Bank 形状：", model.model.memory_bank.shape)

In [ ]:
engine.fit(
    model=model,
    datamodule=datamodule,
)

print("PatchCore 拟合完成！")


In [ ]:
import anomalib.models.components.sampling.k_center_greedy as kcg

original_tqdm = kcg.tqdm

def tqdm_disabled(iterable=None, *args, **kwargs):
    kwargs["disable"] = True
    return original_tqdm(iterable, *args, **kwargs)

kcg.tqdm = tqdm_disabled

print("Coreset 进度条已关闭！")

In [ ]:
from anomalib.engine import Engine

engine = Engine(
    enable_progress_bar=False,
    logger=False,
)

print("Engine 创建成功！")

In [ ]:
from anomalib.data import MVTecAD

datamodule = MVTecAD(
    root="../data/mvtec_ad",
    category="bottle",
    train_batch_size=16,
    eval_batch_size=16,
    num_workers=0,
)

datamodule.setup()

print("Bottle 数据加载成功！")
print("训练图片：", len(datamodule.train_data))
print("测试图片：", len(datamodule.test_data))

In [ ]:
print("Backbone:", model.model.backbone)

In [ ]:
print("模型创建成功！")
print(type(model))

In [ ]:
from anomalib.models import Patchcore

model = Patchcore(
    backbone="wide_resnet50_2",
    layers=["layer2", "layer3"],
    coreset_sampling_ratio=0.1,
)

print(model)

In [ ]:
# 取一张正常图片
normal_path = sorted(normal_dir.glob("*.png"))[0]

# 取一张 broken_large 异常图片
broken_path = sorted(broken_dir.glob("*.png"))[0]

# 找到它对应的 Ground Truth
mask_path = mask_dir / f"{broken_path.stem}_mask.png"

# 读取图片
normal_img = Image.open(normal_path)
broken_img = Image.open(broken_path)
mask_img = Image.open(mask_path)

# 并排显示
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(normal_img)
axes[0].set_title("Normal - Train")

axes[1].imshow(broken_img)
axes[1].set_title("Anomaly - Broken Large")

axes[2].imshow(mask_img, cmap="gray")
axes[2].set_title("Ground Truth Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Notebook 位于 notebooks/，所以 .. 回到项目根目录
data_dir = Path("../data/mvtec_ad/bottle")

normal_dir = data_dir / "train" / "good"
broken_dir = data_dir / "test" / "broken_large"
mask_dir = data_dir / "ground_truth" / "broken_large"

print("正常训练图片数量：", len(list(normal_dir.glob("*.png"))))
print("异常测试图片数量：", len(list(broken_dir.glob("*.png"))))
print("Ground Truth 数量：", len(list(mask_dir.glob("*.png"))))